generate tables:
1. Product table: sku, product_name, cost (how much money it costs to make the product), price (how much we charge customers), brand ()
2. Orders: order_id, sku, quantity, date, product_price (how much a customer paid for these products, maybe not needed here? maybe price per unit)
    - maybe date should be in one format but in bigquery we transform to another one?
3. Inventory table: sku, current_stock (product units), expiration_date (expiration of the current stock)
    - assumption: in real world example, this table would be updated regularly. for simplicity for this project, our table is static and a "snapshot" for this particular date.

sku is the only unique variable here. product name MIGHT be the same

In [ ]:
# libraries
import pandas as pd
import numpy as np
import random
import string
import itertools

Product Table

In [2]:
# PRODUCT TABLE

# sku: random 150 unique 3 letter string
# product_name: create a list of beauty items (e.g., conditioner, shampoo, etc)
# category: create a 
# brand: create a list of 7 fake brands
# demand: low, medium, high
# price: random(5,25)
# cost: random(0.4-0.7) * price

product_name =[
    ('Shampoo','Haircare'),
    ('Conditioner','Haircare'),
    ('Hair Mask','Haircare'),
    ('Hair Oil','Haircare'),
    ('Hair Serum','Haircare'),
    ('Bond Builder','Haircare'),
    ('Scalp Exfoliator','Haircare'),
    ('Dry Shampoo','Haircare'),
    ('Heat Protectant','Haircare'),
    ('Hairspray','Haircare'),
    ('Gel','Haircare'),
    ('Sea Salt Spray','Haircare'),
    ('Mousse','Haircare'),
    ('Cleanser','Skincare'),
    ('Toner','Skincare'),
    ('Serum','Skincare'),
    ('Eye Cream','Skincare'),
    ('Moisturiser','Skincare'),
    ('Sunscreen','Skincare'),
    ('Face Mask','Skincare'),
    ('Lip Balm','Skincare'),
    ('Spot Treatment','Skincare'),
    ('Exfoliator','Skincare'),
    ('Shower Gel','Bodycare'),
    ('Shower Oil','Bodycare'),
    ('Bar Soap','Bodycare'),
    ('Body Scrub','Bodycare'),
    ('Lotion','Bodycare'),
    ('Body Butter','Bodycare'),
    ('Body Oil','Bodycare'),
    ('Foot Cream','Bodycare'),
    ('Deodorant','Bodycare'),
    ('Razor','Bodycare'),
    ('Shaving Cream','Bodycare'),
    ('Loofah','Bodycare'),
    ('Bath Bomb','Bodycare'),
    ('Bath Salt','Bodycare'),
    ('Body Serum','Bodycare'),
    ('Acetone','Nailcare'),
    ('Nail Strengthener','Nailcare'),
    ('Cuticle Oil','Nailcare'),
    ('Nail File','Nailcare'),
    ('Nail Clippers','Nailcare'),
    ('Cuticle Pusher','Nailcare'),
    ('Cuticle Remover','Nailcare'),
    ('Nail Polish','Nailcare')
]

brand = ['LuxBeauty','EurBeauty','CareForSelf','LovelyBeauty','GlowAndShow','RenewBeauty','Skinore','PamperBeauty']

demand = ['High', 'Medium', 'Low']

cost_fracts = [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]

# generate a unique list of 150 skus. we take the alphabet, generate all unique combinations of 3 letters, return 15 random ones
            # generated_sku_nums = np.random.choice(range(100000,199999), size=150, replace=False) # replace makes sure we only get unique numbers
            # sku_list = ['ORD' + str(num) for num in generated_sku_nums] THIS IS ORDER LIST!!!

alph = string.ascii_uppercase
all_sku_combs = [''.join(let) for let in itertools.product(alph, repeat=3)]
sku_list = random.sample(all_sku_combs, k=150)

# for each unique sku, we select a random product+category pair and assign other values. this means that a product+category pair can be assigned to several different skus
generated_products = []
i = 0
for sku in sku_list: 
    price = random.choice(range(5,26))
    cost = price * random.choice(cost_fracts)
    prod,categ = random.choice(product_name)
    product = {
        'sku': sku,
        # product_name = [('a','b'),('a','b'),('a','b')]
        'product_name': prod , # random first string from the product_name list
        'category': categ, # take the second string from the product_name pair
        'brand': random.choice(brand),
        'demand': random.choice(demand),
        'price': float(price), # float in order to match cost
        'cost': round(cost,2)
    }
    generated_products.append(product)

products_df = pd.DataFrame(generated_products)
#print(products)
products_df

,sku,product_name,category,brand,demand,price,cost
0,OHU,Heat Protectant,Haircare,LuxBeauty,High,23.0,14.95
1,LXE,Body Butter,Bodycare,GlowAndShow,High,15.0,6.75
2,ZVE,Body Oil,Bodycare,LuxBeauty,Low,14.0,5.60
3,FNE,Razor,Bodycare,Skinore,Medium,16.0,11.20
4,KTI,Loofah,Bodycare,LuxBeauty,High,8.0,5.60
...,...,...,...,...,...,...,...
145,QKJ,Shower Gel,Bodycare,Skinore,Medium,12.0,7.80
146,LAY,Cuticle Remover,Nailcare,LuxBeauty,High,11.0,5.50
147,EML,Foot Cream,Bodycare,LovelyBeauty,Low,21.0,10.50
148,GBV,Cuticle Pusher,Nailcare,GlowAndShow,Low,21.0,12.60


Orders Table

In [38]:
# ORDER TABLE

# order_id: OR + random 9 unique numbers; generate 10k orders
            # generated_sku_nums = np.random.choice(range(100000,199999), size=10000, replace=False) # replace makes sure we only get unique numbers
            # sku_list = ['ORD' + str(num) for num in generated_sku_nums] THIS IS ORDER LIST!!!

# sku: = product sku BUT if demand=high then take those skus more frequently, if medium then medium, etc.
# quantity: random(1,11), but skew 70% to be < 5.
# date: random dates from 2025-06-01 to 2026-06-01
# unit_product_price: = price

#order_id
generated_order_nums = np.random.choice(range(100000,199999), size=10000, replace=False) # replace makes sure we only get unique numbers
order_list = ['ORD' + str(num) for num in generated_order_nums]

#skus based on demand
high_dem_sku = products_df[products_df['demand'] == 'High']['sku'].tolist() #select sku from products_df where demand='High'
medium_dem_sku = products_df[products_df['demand'] == 'Medium']['sku'].tolist()
low_dem_sku = products_df[products_df['demand'] == 'Low']['sku'].tolist()


orders_final_list = []
for ord in order_list:
    # for each order, we "randomly" select which demand bucket we want. depending on the outcome, we take a random sku from that bucket, and that becomes our item for the order
    random_demand = random.choices(['High', 'Medium', 'Low'], weights=[0.7,0.2,0.1])[0]
    if random_demand == 'High':
        sku = random.choice(high_dem_sku)
    elif random_demand == 'Medium':
        sku = random.choice(medium_dem_sku)
    else:
        sku = random.choice(low_dem_sku)

    random_quant = random.choices(['small', 'big'], weights = [0.7, 0.3])[0]
#    random_quant = random.choices([random.choice(range(1,5)),random.choice(range(5,))], weights = [0.7, 0.3])[0]

    if random_quant == 'small':
        quantity = random.choice(range(1,5))
    else:
        quantity = random.choice(range(5,11))
    
    #unit_price = products_df[products_df['sku']==sku]['price']
    date_range = pd.date_range('2025-06-01', '2026-06-01')

    orders = {
        'order_id': ord,
        'sku': sku,
        'quantity':quantity,
        'order_date': random.choice(date_range), # later change: only select the date, no hours minutes etc. Actually we can leave it here and change it in bigquery
        'unit_product_price': products_df[products_df['sku']==sku]['price'].iloc[0],
        'demand': random_demand

    }

    orders_final_list.append(orders)


#orders_final_list[:10]
# 'sku': random.choices()

# we split the skus from the product table into three lists based on demand and when creating order rows, we assign weights/probabilities based on the demand (0.7,0.2,0.1)
orders_df = pd.DataFrame(orders_final_list)
orders_df

,order_id,sku,quantity,order_date,unit_product_price,demand
0,ORD165570,RSQ,1,2025-06-19,8.0,Medium
1,ORD130090,NGJ,10,2026-04-12,7.0,High
2,ORD145518,MBS,2,2026-05-27,22.0,High
3,ORD173331,CJR,2,2026-05-03,5.0,Medium
4,ORD173839,NLD,3,2026-04-27,9.0,High
...,...,...,...,...,...,...
9995,ORD175686,UCZ,3,2025-12-15,14.0,High
9996,ORD161201,TAY,6,2025-09-09,11.0,Medium
9997,ORD169555,LFI,1,2025-08-01,9.0,Medium
9998,ORD162314,OHU,2,2026-01-05,23.0,High


Inventory Table

In [ ]:
# INVENTORY TABLE

# sku: = product sku
# current_stock: if price < 15 then random(300, 500)
                # else: random (100, 300)
# expiration_date: random from 2026-06-01 to 2026-12-31 BUT 70% of expiration dates should be in the range 2026-06-01 - 2026-10-01


# the inventory should be done for every product (sku), despite it having sales or not

early_exp_size = int(0.7 * len(products_df['sku']))
late_exp_size = int(len(products_df['sku'])) - early_exp_size
# generate 0.7 * n dates (not necessarily unique) in range 2026-06-01 - 2026-10-01, then generate the rest 2026-10-02-2026-12-31. make them into a list and use random choice
early_exp_list = np.random.choice(pd.date_range('2026-06-01','2026-09-30'), size = early_exp_size, replace=True)
late_exp_list = np.random.choice(pd.date_range('2026-10-01','2026-12-31'), size = late_exp_size)
all_exp_dates = np.concatenate([early_exp_list, late_exp_list])
np.random.shuffle(all_exp_dates)

inv_stock_list = []
for _, row in products_df.iterrows():
    if row['price'] > 15:
        cur_stock = np.random.choice(range(100,300))
    else:
        cur_stock = np.random.choice(range(300,500))
    inv_stock_list.append(cur_stock)

inventory = {
    'sku': products_df['sku'],
    'current_stock': inv_stock_list, #cur_stock,
    'expiration_date': all_exp_dates
}

inventory_df = pd.DataFrame(inventory)
inventory_df

,sku,current_stock,expiration_date
0,OHU,283,2026-11-11
1,LXE,487,2026-11-30
2,ZVE,390,2026-08-17
3,FNE,206,2026-11-28
4,KTI,465,2026-11-14
...,...,...,...
145,QKJ,497,2026-10-01
146,LAY,421,2026-10-17
147,EML,265,2026-07-18
148,GBV,290,2026-08-23


Save Files as CSV Files

In [41]:
products_df.to_csv('products.csv', index=False)
orders_df.to_csv('orders.csv', index = False)
inventory_df.to_csv('inventory.csv', index=False)

future addition: quantity. some products might be bundles, for instance buying 3 shampoos are cheaper. so shampoo sku: smp, shampoo 3 bundle: smp3.